# 02 — Evaluation & Statistical Analysis
**Paper:** *Markov Logic Process: Augmenting Reinforcement Learning with Symbolic
Association-Rule Reasoning via the Logos Module*
**Authors:** Saiyam Jain · Swaroop Bhowmik · Dipanjan Choudhury · Santosh Kumar Sahoo

### What this notebook does
Loads all per-seed results from `checkpoints/seed_ckpts/`, computes:
- Table I: Main results (Final MA-100, Eval Mean, Episodes-to-Solve) with Welch t-tests
- Table II: Post-convergence stability statistics
- Figure 2: Ablation bar chart (incremental Logos component contributions)
- Figure 3: Learning curves (mean ± 1σ) with significance annotations
- Figure 4: Evaluation reward violin plots
- Figure 5: Convergence speed box plots + post-convergence scatter
All figures saved as high-res PNG and PDF in `checkpoints/figures/`.

## 0 · Imports & Load Results

In [ ]:
import os, sys, json, warnings
from pathlib import Path
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats as scipy_stats
from collections import defaultdict

warnings.filterwarnings('ignore')
matplotlib.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 150, 'savefig.dpi': 300,
    'savefig.bbox': 'tight', 'savefig.pad_inches': 0.05,
})

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE  = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
PATHS = {
    'ckpt':   BASE/'checkpoints',
    'seeds':  BASE/'checkpoints'/'seed_ckpts',
    'models': BASE/'checkpoints'/'models',
    'figs':   BASE/'checkpoints'/'figures',
}
for p in PATHS.values(): p.mkdir(parents=True, exist_ok=True)

EXPERIMENT_ENVS = {
    'll_std':  {'display_name': 'LunarLander (standard)'},
    'll_wind': {'display_name': 'LunarLander (wind + turbulence)'},
}
AGENT_TYPES = ['MDP', 'MLP-Bellman', 'MLP-Full']
AGENT_COLORS = {'MDP': '#6c757d', 'MLP-Bellman': '#2196F3', 'MLP-Full': '#4CAF50'}
AGENT_LABELS = {'MDP': 'MDP (baseline)', 'MLP-Bellman': 'MLP-Bellman (ablation)', 'MLP-Full': 'MLP-Full (proposed)'}
SEEDS = [42, 123, 456, 789, 1337]
SOLVE_THR = 200

# ── Load seed results ─────────────────────────────────────────────────────────
ALL_RESULTS = {}
for p in sorted(PATHS['seeds'].glob('*.json')):
    parts = p.stem.split('__')
    if len(parts) != 3: continue
    ek, sa, sp = parts
    at = sa.replace('_','-'); seed = int(sp.replace('seed',''))
    with open(p) as f: res = json.load(f)
    ALL_RESULTS.setdefault(ek,{}).setdefault(at,{})[seed] = res

n_loaded = sum(len(v2) for v in ALL_RESULTS.values() for v2 in v.values())
print(f"Loaded {n_loaded} seed results from {PATHS['seeds']}")

## 1 · Statistical Utilities

In [ ]:
def compute_ma(rewards, w=100):
    arr = np.array(rewards, float)
    if len(arr) < w: return np.full(len(arr), np.nan)
    return np.convolve(arr, np.ones(w)/w, mode='valid')

def bootstrap_ci(vals, n=2000, alpha=0.05, rng=None):
    rng = rng or np.random.default_rng(0)
    means = [rng.choice(vals, len(vals), replace=True).mean() for _ in range(n)]
    return np.percentile(means, [100*alpha/2, 100*(1-alpha/2)])

def welch_ttest(a, b):
    a_c = [v for v in a if np.isfinite(v)]; b_c = [v for v in b if np.isfinite(v)]
    if len(a_c)<2 or len(b_c)<2: return dict(t=None, p=None, d=None, stars='')
    t, p = scipy_stats.ttest_ind(b_c, a_c, equal_var=False)
    s = np.sqrt(((len(a_c)-1)*np.std(a_c,ddof=1)**2+(len(b_c)-1)*np.std(b_c,ddof=1)**2)/(len(a_c)+len(b_c)-2+1e-12))
    d = (np.mean(b_c)-np.mean(a_c))/(s+1e-12)
    stars = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else ''))
    return dict(t=float(t), p=float(p), d=float(d), stars=stars)

def ep_to_solve(rewards, thr=200, w=100):
    ma = compute_ma(rewards, w)
    idx = np.where(ma>=thr)[0]
    return float(idx[0]+w) if len(idx) else float('inf')

def post_conv_stats(rewards, thr=200, w=100):
    ma = compute_ma(rewards, w)
    idx = np.where(ma>=thr)[0]
    if not len(idx): return None, None, None
    mpa = ma[idx[0]:]
    return float(mpa.mean()), float(mpa.std(ddof=1)) if len(mpa)>1 else 0.0, float(mpa.min())

print("Statistical utilities defined ✅")

## 2 · Table I — Main Results

In [ ]:
print("TABLE I — Main Results (Mean ± Std, N=5 seeds)")
print("Significance vs MDP baseline (Welch t-test): * p<.05  ** p<.01  *** p<.001")
print()

# Header
hdr = f"{'Env':<10} {'Agent':<15} {'Final MA-100':>13} {'Eval Mean':>12} {'Ep to Solve':>12}"
print(hdr); print('═'*len(hdr))

table1_data = {}
for ek, ecfg in EXPERIMENT_ENVS.items():
    print(f"  {ecfg['display_name']}")
    mdp_evals   = [ALL_RESULTS.get(ek,{}).get('MDP',{}).get(s,{}).get('eval_mean', np.nan) for s in SEEDS]
    mdp_ma100   = [ALL_RESULTS.get(ek,{}).get('MDP',{}).get(s,{}).get('final_ma100', np.nan) for s in SEEDS]
    mdp_solves  = [ALL_RESULTS.get(ek,{}).get('MDP',{}).get(s,{}).get('ep_to_solve', np.inf) for s in SEEDS]

    for at in AGENT_TYPES:
        seeds_r = ALL_RESULTS.get(ek,{}).get(at,{})
        if not seeds_r: print(f"  {'':10} {at:<15} {'—':>13} {'—':>12} {'—':>12}"); continue

        evals  = [seeds_r.get(s,{}).get('eval_mean', np.nan)   for s in SEEDS]
        ma100s = [seeds_r.get(s,{}).get('final_ma100', np.nan) for s in SEEDS]
        solves = [seeds_r.get(s,{}).get('ep_to_solve', np.inf) for s in SEEDS]

        sig_eval  = welch_ttest(mdp_evals,  evals)['stars']  if at!='MDP' else ''
        sig_ma100 = welch_ttest(mdp_ma100, ma100s)['stars']  if at!='MDP' else ''

        finite_s = [v for v in solves if np.isfinite(v)]
        solve_str = f"{np.mean(finite_s):.1f}±{np.std(finite_s):.1f}" if finite_s else "∞"

        row = (f"  {ek:<10} {at:<15}"
               f"  {np.nanmean(ma100s):>6.1f}±{np.nanstd(ma100s):>5.1f}{sig_ma100:<3}"
               f"  {np.nanmean(evals):>6.1f}±{np.nanstd(evals):>5.1f}{sig_eval:<3}"
               f"  {solve_str:>12}")
        print(row)

        table1_data[(ek,at)] = dict(
            evals=evals, ma100s=ma100s, solves=solves,
            eval_mean=np.nanmean(evals), eval_std=np.nanstd(evals),
            ma100_mean=np.nanmean(ma100s), solve_mean=np.nanmean(finite_s) if finite_s else np.inf,
        )
    print()

## 3 · Table II — Post-Convergence Stability

In [ ]:
print("TABLE II — Post-Convergence Stability (Mean ± Std, episodes after first MA-100 ≥ 200)")
print("Lower std and higher min indicate more stable policies.")
print()
hdr2 = f"{'Env':<10} {'Agent':<15} {'Post-conv Mean':>15} {'Post-conv Std':>14} {'Post-conv Min':>14}"
print(hdr2); print('═'*len(hdr2))

table2_data = {}
for ek, ecfg in EXPERIMENT_ENVS.items():
    print(f"  {ecfg['display_name']}")
    for at in AGENT_TYPES:
        seeds_r = ALL_RESULTS.get(ek,{}).get(at,{})
        if not seeds_r: continue
        pc_means, pc_stds, pc_mins = [], [], []
        for s in SEEDS:
            rews = seeds_r.get(s,{}).get('ep_rewards',[])
            if not rews: continue
            m, st, mn = post_conv_stats(rews)
            if m is not None: pc_means.append(m); pc_stds.append(st); pc_mins.append(mn)
        if not pc_means:
            print(f"  {ek:<10} {at:<15}  {'—':>15} {'—':>14} {'—':>14}"); continue
        print(f"  {ek:<10} {at:<15}  "
              f"{np.mean(pc_means):>7.1f}±{np.std(pc_means):>4.1f}    "
              f"{np.mean(pc_stds):>7.1f}±{np.std(pc_stds):>4.1f}    "
              f"{np.mean(pc_mins):>7.1f}±{np.std(pc_mins):>4.1f}")
        table2_data[(ek,at)] = dict(pc_means=pc_means, pc_stds=pc_stds, pc_mins=pc_mins)
    print()

## 4 · Figure 2 — Ablation Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)

for ax, (ek, ecfg) in zip(axes, EXPERIMENT_ENVS.items()):
    mdp_mean = table1_data.get((ek,'MDP'), {}).get('eval_mean', 0)
    x_pos = np.arange(len(AGENT_TYPES))
    bar_means, bar_cis = [], []
    for at in AGENT_TYPES:
        evals = [v for v in table1_data.get((ek,at),{}).get('evals',[]) if np.isfinite(v)]
        if not evals: bar_means.append(0); bar_cis.append((0,0)); continue
        ci = bootstrap_ci(np.array(evals))
        bar_means.append(np.mean(evals)); bar_cis.append(ci)

    colors = [AGENT_COLORS[at] for at in AGENT_TYPES]
    bars = ax.bar(x_pos, bar_means, color=colors, alpha=0.85, edgecolor='white',
                  linewidth=1.2, width=0.55, zorder=3)
    for i, (bar, ci) in enumerate(zip(bars, bar_cis)):
        ax.errorbar(x_pos[i], bar_means[i],
                    yerr=[[bar_means[i]-ci[0]], [ci[1]-bar_means[i]]],
                    fmt='none', color='#333', capsize=5, capthick=1.5, zorder=5)
    # Percentage labels
    for i, (at, m) in enumerate(zip(AGENT_TYPES, bar_means)):
        if at != 'MDP' and mdp_mean > 0:
            pct = (m - mdp_mean) / abs(mdp_mean) * 100
            ax.text(x_pos[i], m + (max(bar_means)*0.02), f'+{pct:.1f}%',
                    ha='center', va='bottom', fontsize=9, fontweight='bold', color='#333')
    # Contribution arrows
    if len(bar_means) == 3 and all(bar_means):
        diff_bell = bar_means[1]-bar_means[0]
        diff_full = bar_means[2]-bar_means[1]
        ax.annotate('', xy=(x_pos[1], bar_means[0]+diff_bell*0.5),
                    xytext=(x_pos[0], bar_means[0]+diff_bell*0.5),
                    arrowprops=dict(arrowstyle='->', color='#555', lw=1.3))
        ax.annotate('', xy=(x_pos[2], bar_means[1]+diff_full*0.5),
                    xytext=(x_pos[1], bar_means[1]+diff_full*0.5),
                    arrowprops=dict(arrowstyle='->', color='#555', lw=1.3))

    ax.set_xticks(x_pos)
    ax.set_xticklabels([AGENT_LABELS[at] for at in AGENT_TYPES], rotation=12, ha='right', fontsize=9)
    ax.set_title(ecfg['display_name'], fontweight='bold', fontsize=11)
    ax.set_ylabel('Eval Mean Reward ± 95% Bootstrap CI')
    ax.grid(axis='y', alpha=0.3, zorder=0)
    ax.set_ylim(bottom=max(0, min(bar_means)*0.85))

fig.suptitle('Figure 2 — Ablation: Incremental Contribution of Each Logos Component', fontweight='bold', y=1.01)
plt.tight_layout()
out = PATHS['figs']/'fig2_ablation_bars.pdf'
plt.savefig(out); plt.savefig(str(out).replace('.pdf','.png'))
plt.show(); print(f"Saved: {out}")

## 5 · Figure 3 — Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, (ek, ecfg) in zip(axes, EXPERIMENT_ENVS.items()):
    for at in AGENT_TYPES:
        seeds_r = ALL_RESULTS.get(ek,{}).get(at,{})
        if not seeds_r: continue
        all_ma = []
        for s in SEEDS:
            rews = seeds_r.get(s,{}).get('ep_rewards',[])
            if rews: all_ma.append(compute_ma(rews))
        if not all_ma: continue
        min_len = min(len(m) for m in all_ma)
        arr = np.array([m[:min_len] for m in all_ma])
        ep_ax = np.arange(min_len) + 100   # MA starts at episode 100
        mean_ = arr.mean(0); std_ = arr.std(0)
        color = AGENT_COLORS[at]
        ax.plot(ep_ax, mean_, color=color, lw=2, label=AGENT_LABELS[at], zorder=4)
        ax.fill_between(ep_ax, mean_-std_, mean_+std_, color=color, alpha=0.15, zorder=2)

    ax.axhline(SOLVE_THR, color='red', ls='--', lw=1.2, alpha=0.7, label='Solve threshold (200)')
    # Significance annotation
    mdp_m  = table1_data.get((ek,'MDP'),{}).get('ma100_mean',0)
    full_m = table1_data.get((ek,'MLP-Full'),{}).get('ma100_mean',0)
    full_e = table1_data.get((ek,'MLP-Full'),{}).get('evals',[])
    mdp_e  = table1_data.get((ek,'MDP'),{}).get('evals',[])
    sig = welch_ttest(mdp_e, full_e)
    ax.set_title(f"{ecfg['display_name']}\nMLP-Full vs MDP: p{sig['stars'] if sig['stars'] else '=n.s.'}  d={sig['d']:.2f}" if sig['d'] else ecfg['display_name'],
                 fontweight='bold', fontsize=10)
    ax.set_xlabel('Episode'); ax.set_ylabel('MA-100 Reward')
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(alpha=0.3, zorder=0)

fig.suptitle('Figure 3 — Learning Curves (mean ± 1σ, N=5 seeds per agent)', fontweight='bold', y=1.01)
plt.tight_layout()
out = PATHS['figs']/'fig3_learning_curves.pdf'
plt.savefig(out); plt.savefig(str(out).replace('.pdf','.png'))
plt.show(); print(f"Saved: {out}")

## 6 · Figure 4 — Evaluation Reward Violin Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (ek, ecfg) in zip(axes, EXPERIMENT_ENVS.items()):
    all_data, positions, colors_v = [], [], []
    for i, at in enumerate(AGENT_TYPES):
        seeds_r = ALL_RESULTS.get(ek,{}).get(at,{})
        pooled = []
        for s in SEEDS:
            pooled += seeds_r.get(s,{}).get('eval_rewards',[])
        if pooled:
            all_data.append(pooled); positions.append(i+1); colors_v.append(AGENT_COLORS[at])

    if all_data:
        parts = ax.violinplot(all_data, positions=positions, showmedians=True,
                               showmeans=False, showextrema=True)
        for i, (pc, col) in enumerate(zip(parts['bodies'], colors_v)):
            pc.set_facecolor(col); pc.set_alpha(0.6)
        for elem in ['cbars','cmins','cmaxes']:
            parts[elem].set_color('#555')
        parts['cmedians'].set_color('#333'); parts['cmedians'].set_linewidth(2)
        # Per-seed mean dots
        for i, at in enumerate(AGENT_TYPES):
            seeds_r = ALL_RESULTS.get(ek,{}).get(at,{})
            for s in SEEDS:
                er = seeds_r.get(s,{}).get('eval_rewards',[])
                if er: ax.scatter(i+1, np.mean(er), s=25, color=AGENT_COLORS[at],
                                   edgecolors='white', linewidth=0.8, zorder=6, alpha=0.85)

    ax.axhline(SOLVE_THR, color='red', ls='--', lw=1.2, alpha=0.7)
    ax.set_xticks(range(1, len(AGENT_TYPES)+1))
    ax.set_xticklabels([AGENT_LABELS[at] for at in AGENT_TYPES], rotation=12, ha='right', fontsize=9)
    ax.set_title(ecfg['display_name'], fontweight='bold')
    ax.set_ylabel('Episode Reward (50 greedy eval episodes per seed)')
    ax.grid(axis='y', alpha=0.3, zorder=0)

fig.suptitle('Figure 4 — Evaluation Reward Distributions\n(dots = per-seed means, dashed = solve threshold)',
             fontweight='bold', y=1.02)
plt.tight_layout()
out = PATHS['figs']/'fig4_eval_violins.pdf'
plt.savefig(out); plt.savefig(str(out).replace('.pdf','.png'))
plt.show(); print(f"Saved: {out}")

## 7 · Figure 5 — Convergence Speed & Post-Convergence Stability

In [ ]:
fig = plt.figure(figsize=(15,6))
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)
ax_l, ax_r = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])

# ── Left: Episodes-to-solve box plots ────────────────────────────────────────
all_solves_by_agent = {at: [] for at in AGENT_TYPES}
for ek in EXPERIMENT_ENVS:
    for at in AGENT_TYPES:
        seeds_r = ALL_RESULTS.get(ek,{}).get(at,{})
        for s in SEEDS:
            rews = seeds_r.get(s,{}).get('ep_rewards',[])
            if rews:
                v = ep_to_solve(rews)
                if np.isfinite(v): all_solves_by_agent[at].append(v)

x_positions = np.arange(len(AGENT_TYPES))+1
bp = ax_l.boxplot(
    [all_solves_by_agent.get(at,[0]) for at in AGENT_TYPES],
    positions=x_positions, widths=0.5, patch_artist=True,
    medianprops=dict(color='white', lw=2),
)
for patch, at in zip(bp['boxes'], AGENT_TYPES):
    patch.set_facecolor(AGENT_COLORS[at]); patch.set_alpha(0.75)
for at, xp in zip(AGENT_TYPES, x_positions):
    for v in all_solves_by_agent.get(at,[]):
        ax_l.scatter(xp + np.random.uniform(-0.12,0.12), v, s=22,
                     color=AGENT_COLORS[at], alpha=0.7, zorder=5)
ax_l.set_xticks(x_positions)
ax_l.set_xticklabels([AGENT_LABELS[at] for at in AGENT_TYPES], rotation=13, ha='right', fontsize=9)
ax_l.set_ylabel('First episode where MA-100 ≥ 200'); ax_l.set_title('Convergence Speed (lower = better)', fontweight='bold')
ax_l.grid(axis='y', alpha=0.3)

# ── Right: Post-convergence scatter ──────────────────────────────────────────
for ek, ecfg in EXPERIMENT_ENVS.items():
    mrkr = 'o' if 'std' in ek else 's'
    for at in AGENT_TYPES:
        seeds_r = ALL_RESULTS.get(ek,{}).get(at,{})
        pc_m, pc_s = [], []
        for s in SEEDS:
            rews = seeds_r.get(s,{}).get('ep_rewards',[])
            if rews:
                m, st, _ = post_conv_stats(rews)
                if m is not None: pc_m.append(m); pc_s.append(st)
        if pc_m:
            ax_r.scatter(pc_s, pc_m, s=60, color=AGENT_COLORS[at], alpha=0.7,
                         marker=mrkr, edgecolors='white', linewidth=0.8,
                         label=f"{AGENT_LABELS[at]} ({'Std' if 'std' in ek else 'Wind'})", zorder=5)

ax_r.set_xlabel('Post-convergence MA-100 Std (lower = more stable)')
ax_r.set_ylabel('Post-convergence MA-100 Mean (higher = better)')
ax_r.set_title('Stability After Convergence
(upper-left is optimal)', fontweight='bold')
ax_r.legend(fontsize=7, loc='lower left', ncol=1)
ax_r.grid(alpha=0.3)

fig.suptitle('Figure 5 — Convergence Speed (left) and Post-Convergence Stability (right)', fontweight='bold', y=1.01)
plt.tight_layout()
out = PATHS['figs']/'fig5_convergence_stability.pdf'
plt.savefig(out); plt.savefig(str(out).replace('.pdf','.png'))
plt.show(); print(f"Saved: {out}")

## 8 · Export Results JSON

In [ ]:
import json

def _serial(o):
    if isinstance(o,(np.integer,)): return int(o)
    if isinstance(o,(np.floating,)): return float(o)
    if isinstance(o,np.ndarray): return o.tolist()
    if isinstance(o,(frozenset,set)): return sorted(list(o))
    if o==float('inf'): return 'inf'
    raise TypeError(type(o).__name__)

# Build summary dict
summary = {}
for ek in EXPERIMENT_ENVS:
    summary[ek] = {}
    for at in AGENT_TYPES:
        seeds_r = ALL_RESULTS.get(ek,{}).get(at,{})
        evals   = [seeds_r.get(s,{}).get('eval_mean', np.nan)    for s in SEEDS]
        ma100s  = [seeds_r.get(s,{}).get('final_ma100', np.nan)  for s in SEEDS]
        solves  = [seeds_r.get(s,{}).get('ep_to_solve', np.inf)  for s in SEEDS]
        mdp_e   = [ALL_RESULTS.get(ek,{}).get('MDP',{}).get(s,{}).get('eval_mean',np.nan) for s in SEEDS]
        sig     = welch_ttest(mdp_e, evals) if at!='MDP' else {}
        summary[ek][at] = {
            'eval_mean':  float(np.nanmean(evals)),
            'eval_std':   float(np.nanstd(evals)),
            'ma100_mean': float(np.nanmean(ma100s)),
            'ma100_std':  float(np.nanstd(ma100s)),
            'ep_to_solve_mean': float(np.nanmean([v for v in solves if np.isfinite(v)])) if any(np.isfinite(v) for v in solves) else 'inf',
            'n_seeds':    sum(1 for v in evals if np.isfinite(v)),
            'welch_p':    sig.get('p'),
            'cohen_d':    sig.get('d'),
            'sig_stars':  sig.get('stars',''),
        }

out_json = PATHS['ckpt']/'results_summary.json'
with open(out_json,'w') as f: json.dump(summary, f, indent=2, default=_serial)
print(f"Saved: {out_json}")
print()
print("✅ Evaluation complete.")
print(f"✅ Figures saved in: {PATHS['figs']}")
print("✅ Run 03_video.ipynb to generate side-by-side comparison videos.")